# SEC NIL Valuation Demo

End-to-end demo of the SEC football NIL valuation model. One player per position, predictions compared against their public On3 valuation.

**Data quality caveat.** On3 valuations are model-based estimates, not transaction records. Treat reported errors as illustrative, not ground truth. The training sample is small (roughly 50 to 150 labeled SEC players across the five positions covered). k-fold cross-validation is used in place of a single train and test split for that reason.

## Requirements before running

1. `data/cache/cfbd_rosters_stats.json` produced by `python -m data.ingest.cfbd_client ...`.
2. `data/cache/on3_manual.csv` populated with public NIL valuations.
3. `data/cache/knight_newhouse_sec.csv` downloaded.
4. `data/cache/team_records.csv` populated.
5. `python -m data.build_dataset --output data/sec_2025.csv` has been run.
6. `python training/train.py --config config/sec_football_config.yaml` has been run.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from data.preprocessing import load_dataset_csv, filter_trainable
from training.evaluate import load_checkpoint, predict_usd, metrics

DATASET_CSV = '../data/sec_2025.csv'
CHECKPOINT = '../training/checkpoints/best.pt'

## 1. Load the trained model and dataset

In [ ]:
model, state, cfg = load_checkpoint(CHECKPOINT)
df = load_dataset_csv(DATASET_CSV)
print(f'Loaded {len(df)} rows, {df["nil_value_usd"].notna().sum()} labeled')
df['position'].value_counts()

## 2. Overall and per-position metrics

In [ ]:
labeled = filter_trainable(df)
preds = predict_usd(model, state, labeled)
actuals = labeled['nil_value_usd'].to_numpy()

overall = metrics(preds, actuals)
print('overall:', overall)
rows = []
for pos in state.position_order:
    mask = (labeled['position'] == pos).to_numpy()
    m = metrics(preds[mask], actuals[mask])
    m['position'] = pos
    rows.append(m)
pd.DataFrame(rows)[['position', 'n', 'mae', 'rmse', 'mape']]

## 3. Pick five real SEC players, one per position

Edit the list below to match the players in your `on3_manual.csv`. This notebook does not hardcode names because the set of players with public On3 valuations is dynamic.

In [ ]:
target_players = [
    # (name, position)
    ('Garrett Nussmeier', 'QB'),
    ('Ryan Williams', 'WR'),
    ('Oscar Delp', 'TE'),
    ('Harold Perkins Jr.', 'MLB'),
    ('Malaki Starks', 'S'),
]
rows = []
for name, _pos in target_players:
    match = labeled[labeled['name'].str.lower() == name.lower()]
    if match.empty:
        rows.append({'name': name, 'position': _pos, 'predicted_nil_usd': None,
                     'actual_nil_usd': None, 'note': 'not found in dataset'})
        continue
    pred = predict_usd(model, state, match)
    for i, r in match.reset_index(drop=True).iterrows():
        rows.append({'name': r['name'], 'school': r['school'], 'position': r['position'],
                     'predicted_nil_usd': float(pred[i]),
                     'actual_nil_usd': float(r['nil_value_usd'])})
pd.DataFrame(rows)

## 4. Error scatter (predicted vs actual)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(actuals, preds, alpha=0.7)
lo = 0
hi = max(float(actuals.max()), float(preds.max())) * 1.1 if len(actuals) else 1.0
ax.plot([lo, hi], [lo, hi], linestyle='--', color='gray')
ax.set_xlabel('Actual NIL USD')
ax.set_ylabel('Predicted NIL USD')
ax.set_title('Predicted vs actual NIL (SEC football, log-target + Huber)')
ax.set_xscale('symlog')
ax.set_yscale('symlog')
plt.show()

## Limitations

- Labels are noisy. On3 valuations are estimates, not transactions.
- Defensive positions (MLB, S) are under-represented in labeled data; their per-position MAE is typically the worst. Per-position reporting above makes that explicit.
- Social follower counts are sparse and snapshot-only. The model imputes missing values with the training median, which biases predictions for high-profile athletes toward the mean.
- School revenue data lags by one to two fiscal years relative to the current season.